# 02 - Convolutional VAE + Gaussian mixture

Learns a 16-dimensional summary of each daily scalogram (128 scales x 24 hours). A Gaussian mixture on those summaries plus the daily price level assigns each day to a market regime.

Pipeline: scalogram -> ConvVAE encoder -> latent mean (mu) + daily price level -> Gaussian mixture -> regime

Methodology notes:
1. Days are split chronologically (60% train, 20% validation, 20% test). The 2023-2024 El Nino falls in the test period.
2. The Gaussian mixture is fitted on training days only and then applied to all days.
3. Scalograms come from the real price with each day's mean removed, so they show only the intraday shape. `real_price_log` uses the log real price and `real_price` the real price in COP/kWh. The daily level (mean and standard deviation of the same series) is added to the latent vector before clustering.
4. Each run creates a folder in `experiments/conv_vae/` with config, history, metrics, tables and figures, and adds one row to `experiments/registry.csv`.

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import torch

import numpy as np
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt
from prettytable import PrettyTable
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

plt.rcParams['font.family'] = 'Arial'
alt.data_transformers.enable('vegafusion')

from src.data import load_scalograms, load_context, chronological_split, ScalogramDataset
from src.models.conv_vae import ConvVAE
from src.training import set_seed, train
from src.clustering import with_level, level_columns, assign_regimes, fit_clusters, order_regimes, summary_metrics, regime_profile, save_regimes
from src.plots import loss_curves, selection_chart, tsne_chart, regime_timeline, plot_reconstructions
from src import experiments as exp

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Experiment configuration

In [ ]:
SCALOGRAM_SET = "real_price"
CLUSTERING = "gmm"
N_REGIMES = 2

CONFIG = {
    "model": "conv_vae",
    "scalogram_set": SCALOGRAM_SET,
    "clustering": CLUSTERING,
    "n_regimes": N_REGIMES,
    "k_range": [2, 3, 4, 5, 6, 7, 8],
    "z_dim": 16,
    "filters": [128, 64, 32],
    "beta": 1.0,
    "kl_warmup_epochs": 0,
    "optimizer": "Adam",
    "lr": 1e-3,
    "batch_size": 64,
    "epochs": 30,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "seed": 42,
}

## 1. Run folder

In [ ]:
SCALOGRAM_DIR = Path("../data/processed/scalograms") / SCALOGRAM_SET
DATA_DIR = Path("../data/processed")

RUN_DIR = exp.create_run(CONFIG["model"], SCALOGRAM_SET, CLUSTERING, N_REGIMES)
exp.save_config(RUN_DIR, CONFIG)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(CONFIG["seed"])
print(f"Run: {exp.run_id(RUN_DIR)}")
print(f"Device: {DEVICE}")

## 2. Data

In [ ]:
images, metadata = load_scalograms(SCALOGRAM_DIR)
context = load_context(DATA_DIR, metadata)
train_idx, val_idx, test_idx = chronological_split(len(images), CONFIG["train_ratio"], CONFIG["val_ratio"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]

IMG_C, IMG_H, IMG_W = images.shape[1:]
print(f"Scalograms ({SCALOGRAM_SET}): {images.shape}, range [{images.min():.3f}, {images.max():.3f}]")
print(f"Daily level columns: {level_columns(metadata)}")
exp.save_table(RUN_DIR, "splits", exp.split_table(context, {"train": train_idx, "val": val_idx, "test": test_idx}))

In [ ]:
train_loader = DataLoader(ScalogramDataset(images, train_idx), batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(ScalogramDataset(images, val_idx), batch_size=CONFIG["batch_size"], shuffle=False)

## 3. Model

In [ ]:
model = ConvVAE(CONFIG["z_dim"], IMG_C, IMG_H, IMG_W, CONFIG["filters"]).to(DEVICE)

with torch.no_grad():
    out_shape = tuple(model(torch.from_numpy(images[:2]).to(DEVICE))[0].shape[1:])
assert out_shape == (IMG_C, IMG_H, IMG_W), f"decoder output {out_shape} does not match input"

N_PARAMS = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Encoder feature map: {model.encoder.feat_shape}")
print(f"Decoder output: {out_shape}")
exp.save_table(RUN_DIR, "hyperparameters", exp.hyperparameter_table(CONFIG, N_PARAMS))

## 4. Training

In [ ]:
history = train(model, train_loader, val_loader, DEVICE, epochs=CONFIG["epochs"], lr=CONFIG["lr"],
                beta=CONFIG["beta"], kl_warmup_epochs=CONFIG["kl_warmup_epochs"])
exp.save_history(RUN_DIR, history)
exp.save_model(RUN_DIR, model)

## 5. Loss curves

In [ ]:
loss_plot = loss_curves(history, components=("total", "recon", "kl"))
loss_plot

In [ ]:
loss_plot.save(exp.figure_path(RUN_DIR, "loss_curves"), scale_factor=2.0)

## 6. Reconstructions

Test-period days.

In [ ]:
model.eval()
show_idx = test_idx[np.linspace(0, len(test_idx) - 1, 8).astype(int)]
with torch.no_grad():
    recon = model(torch.from_numpy(images[show_idx]).to(DEVICE))[0].cpu().numpy()

fig = plot_reconstructions(images[show_idx], recon, [str(d.date()) for d in context["date"].iloc[show_idx]])
fig.savefig(exp.figure_path(RUN_DIR, "reconstructions"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Latent vectors

In [ ]:
latents = model.extract_latents(images, DEVICE)
exp.save_latents(RUN_DIR, latents)
features = with_level(latents, metadata, train_idx)

active = int((latents.var(axis=0) > 0.01).sum())
print(f"Latent vectors: {latents.shape}")
print(f"Final train KL: {history[-1]['train_kl']:.2f}")
print(f"Active dimensions: {active} of {CONFIG['z_dim']}")
print(f"Clustering features (latents + daily level): {features.shape}")

## 8. Regimes with a Gaussian mixture

The GMM uses the latent vector plus the daily price level (mean and standard deviation, z-scored on training days), because the scalograms only carry the intraday shape. It is fitted on the training days only and then assigns every day. k comes from `N_REGIMES`, or from the lowest BIC when it is None. The table shows, for each k on the training days, the BIC, silhouette (higher is better), Davies-Bouldin (lower is better) and Calinski-Harabasz (higher is better). Regimes are numbered by ascending mean real price, so regime 0 is always the cheapest and labels are comparable across models.

In [ ]:
selection, K, labels, probs = assign_regimes(
    features, train_idx, context["precio_real_cop_kwh"].to_numpy(),
    k=N_REGIMES, k_range=CONFIG["k_range"], seed=CONFIG["seed"], method=CLUSTERING,
)
context["regime"] = labels

exp.save_table(RUN_DIR, "k_selection", exp.selection_table(selection))
print(f"Selected k = {K}")

In [ ]:
selection_plot = selection_chart(selection, K)
selection_plot

In [ ]:
selection_plot.save(exp.figure_path(RUN_DIR, "k_selection"), scale_factor=2.0)

## 9. Latent space

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])
plot_idx = np.sort(rng.choice(len(features), size=min(5000, len(features)), replace=False))
coords = TSNE(n_components=2, perplexity=30, random_state=CONFIG["seed"]).fit_transform(features[plot_idx])

tsne_plot = tsne_chart(coords, labels[plot_idx], f"t-SNE of the latent space (k={K})")
tsne_plot

In [ ]:
tsne_plot.save(exp.figure_path(RUN_DIR, "tsne_regimes"), scale_factor=2.0)

## 10. Regime timeline

Shaded bands are El Nino months (ONI >= 0.5). Dashed lines mark the start of the validation and test periods.

In [ ]:
timeline_plot = regime_timeline(context, split_dates, title=f"Detected regimes over time (k={K})")
timeline_plot

In [ ]:
timeline_plot.save(exp.figure_path(RUN_DIR, "regime_timeline"), scale_factor=2.0)

## 11. Regime profile and clustering metrics

In [ ]:
exp.save_table(RUN_DIR, "regime_profile", exp.regime_profile_table(regime_profile(context)))

In [ ]:
metrics = {"run_id": exp.run_id(RUN_DIR), "model": CONFIG["model"], "scalogram_set": SCALOGRAM_SET,
           "clustering": CLUSTERING, "k": K, **summary_metrics(features, labels, seed=CONFIG["seed"])}
save_regimes(RUN_DIR, context, labels, probs, metrics)
exp.save_table(RUN_DIR, "clustering_metrics", exp.metrics_table(metrics))
exp.log_run(RUN_DIR, CONFIG, metrics)

In [ ]:
spread = context.groupby("regime")["precio_real_cop_kwh"].describe(percentiles=[0.1, 0.5, 0.9])
spread[["count", "10%", "50%", "90%", "min", "max"]].round(1)

## 12. Exploratory checks

In [ ]:
price = context["precio_real_cop_kwh"].to_numpy()
level = with_level(np.zeros((len(latents), 0)), metadata, train_idx)
smooth = pd.DataFrame(latents).rolling(7, min_periods=1).mean().to_numpy()

rows = (
    ("level only", level),
    ("latents only", latents),
    ("latents + level", features),
    ("7d-mean latents + level", with_level(smooth, metadata, train_idx)),
)
t = PrettyTable(["Features", "Silhouette", "Davies-Bouldin", "Mean run", "Switches / yr", "Median price by regime"])
for name, f in rows:
    lab = order_regimes(fit_clusters(f, train_idx, K, CLUSTERING, CONFIG["seed"])[0], price)
    m = summary_metrics(f, lab, seed=CONFIG["seed"])
    med = context.assign(r=lab).groupby("r")["precio_real_cop_kwh"].median().round(0).tolist()
    t.add_row([name, f"{m['silhouette']:.3f}", f"{m['davies_bouldin']:.3f}", f"{m['mean_run_days']:.1f}",
               f"{m['switches_per_year']:.0f}", med])
print(t)
print(pd.crosstab(labels, context["date"].dt.dayofweek, normalize="columns").mul(100).round(0))

In [ ]:
print("Latent variance per dim:", np.round(latents.var(axis=0), 5))
print("Latent std range:", latents.std(axis=0).min().round(4), "to", latents.std(axis=0).max().round(4))